In [17]:
import polars as pl
from pathlib import Path
import os

In [21]:
output_dir = Path("../dataset/split_raw")
input_file= Path("../dataset/2019-Oct.csv.gz")

In [22]:
time_boundaries = [
    ("week1", "2019-10-01 00:00:00 UTC", "2019-10-08 00:00:00 UTC"),
    ("week2", "2019-10-08 00:00:00 UTC", "2019-10-15 00:00:00 UTC"),
    ("week3", "2019-10-15 00:00:00 UTC", "2019-10-22 00:00:00 UTC"),
    ("week4", "2019-10-22 00:00:00 UTC", "2019-11-01 00:00:00 UTC"),
]

In [23]:
for name, start_time, end_time in time_boundaries:
    output_file = output_dir / f"2019-Oct-{name}.csv"
    df = pl.scan_csv(input_file)
    
    # Filter the data based on the time boundaries
    filtered_df = df.filter(
        (pl.col("event_time") >= start_time) &
        (pl.col("event_time") < end_time)
    )
    filtered_df.sink_csv(str(output_file))
    
    # Save the filtered data to a new file
    filtered_df.sink_csv(str(output_file))

### recheck

In [37]:
import pandas as pd

In [40]:
# df_week1 = pd.read_csv(output_dir / "2019-Oct-week1.csv")
# df_week1.shape

In [ ]:
df_week1['user_session'].nunique()

957543

In [32]:
!cd ../dataset/split_raw && gzip -f 2019-Oct-week*.csv

In [41]:
df_week1_gz = pd.read_csv(output_dir / "2019-Oct-week1.csv.gz", compression='gzip')
df_week1_gz.shape

(8829315, 9)

In [42]:
df_week1_gz['user_session'].nunique()

1877365

### split 1 day

In [2]:
import polars as pl
from pathlib import Path
import time
import pandas as pd
output_dir = Path("../dataset/split_raw/2019-Oct-Day1.csv")
input_file= Path("../dataset/split_raw/2019-Oct-week1.csv")
start_time = "2019-10-01 00:00:00 UTC"
end_time = "2019-10-02 00:00:00 UTC"

In [10]:
pl.scan_csv(input_file).filter(
    (pl.col("event_time") >= start_time) &
    (pl.col("event_time") < end_time)
).sink_csv(output_dir)

In [5]:
df_day1 = pd.read_csv("../dataset/split_raw/2019-Oct-week1.csv")
df_day1.shape

(8829315, 9)

In [ ]:
# !cd ../dataset/split_raw && gzip -f 2019-Oct-Day1.csv

In [6]:
df_day1.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


### Recheck file train

In [3]:
df_split = pd.read_parquet("../data/gold/session_split_map.parquet")

In [4]:
df_split

,user_session,session_start_time,session_end_time,split
0,0000c738-9fbb-466a-906c-e3b2a62730da,2019-10-01 19:08:39,2019-10-01 19:12:07,test
1,00012d23-c857-40af-b8cb-ada787bc00cc,2019-10-01 15:49:07,2019-10-01 15:49:51,train
2,00016584-60b8-4d68-8b89-960b6a5cb750,2019-10-01 19:24:49,2019-10-01 19:24:49,test
3,00016bb9-b50d-4bcb-95d1-9c375f214c66,2019-10-01 09:51:44,2019-10-01 09:52:14,train
4,0001c1b8-85f4-46c6-837d-f5becd54f208,2019-10-01 22:27:58,2019-10-01 22:27:58,test
...,...,...,...,...
268511,fffe00fd-d83e-46d7-aac9-03bf5a4e1cf1,2019-10-01 20:36:01,2019-10-01 20:39:51,test
268512,fffe42a6-aa9d-456c-929e-5476e8980538,2019-10-01 07:44:02,2019-10-01 07:44:02,train
268513,fffe439b-c02e-4ebd-997c-d0873506341d,2019-10-01 16:02:25,2019-10-01 16:02:25,train
268514,fffe6343-f74f-480f-874a-6ac214c307e7,2019-10-01 17:32:56,2019-10-01 17:38:34,val


In [5]:
df_train = pd.read_parquet("../data/gold/train.parquet")

In [6]:
df_train

,source_event_time,category_id,user_session,user_id,event_type,product_id,category_code,brand,price,total_views,total_carts,net_cart_count,cart_to_view_ratio,unique_categories,unique_products,session_duration_sec,label
0,2019-10-01 15:49:07,2053013565983425517,00012d23-c857-40af-b8cb-ada787bc00cc,515120334,view,3701056,appliances.environment.vacuum,samsung,64.33,0,0,0,0.0,0,0,0.0,0
1,2019-10-01 15:49:51,2053013565983425517,00012d23-c857-40af-b8cb-ada787bc00cc,515120334,view,3700338,appliances.environment.vacuum,samsung,66.90,1,0,0,0.0,1,1,44.0,0
2,2019-10-01 09:51:44,2053013556311359947,00016bb9-b50d-4bcb-95d1-9c375f214c66,523769618,view,12300393,construction.tools.drill,None,39.10,0,0,0,0.0,0,0,0.0,0
3,2019-10-01 09:52:14,2053013556311359947,00016bb9-b50d-4bcb-95d1-9c375f214c66,523769618,view,12300393,construction.tools.drill,None,39.10,1,0,0,0.0,1,1,30.0,0
4,2019-10-01 10:09:56,2053013555631882655,0001d713-f9c4-4d96-8f8c-5da2bff7bbf9,513196971,view,1003306,electronics.smartphone,apple,587.53,0,0,0,0.0,0,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950673,2019-10-01 11:19:54,2053013561847841545,fffceee3-7369-4d8b-8c4b-692640052658,516335020,view,23700097,furniture.bedroom.blanket,askona,74.08,0,0,0,0.0,0,0,0.0,0
950674,2019-10-01 11:21:41,2053013563584283495,fffceee3-7369-4d8b-8c4b-692640052658,516335020,view,26300089,None,lucente,271.82,1,0,0,0.0,1,1,107.0,0
950675,2019-10-01 11:28:59,2053013559792632471,fffceee3-7369-4d8b-8c4b-692640052658,516335020,view,17200216,furniture.living_room.sofa,None,507.07,2,0,0,0.0,2,2,545.0,0
950676,2019-10-01 07:44:02,2053013555631882655,fffe42a6-aa9d-456c-929e-5476e8980538,537794589,view,1003549,electronics.smartphone,samsung,344.64,0,0,0,0.0,0,0,0.0,0
